# A trainable intervention: learn one additive vector by gradient descent

Some interventions are learned parameters rather than fixed edits. This recipe learns a small additive vector at one activation site: each step forks the captured trace, replays it with the vector added at the ReLU, and backpropagates the loss into the vector alone. The model's own parameters stay frozen.

In [ ]:
# recipe-test: tool=torchlens expected=True
from __future__ import annotations

from typing import Any

import torch
from torch import nn

import torchlens as tl


class TinyRegressor(nn.Module):
    """Four inputs, one ReLU layer, two outputs."""

    def __init__(self) -> None:
        super().__init__()
        self.hidden = nn.Linear(4, 4)
        self.head = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(torch.relu(self.hidden(x)))


torch.manual_seed(5)
model = TinyRegressor().eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
x = torch.randn(3, 4)
target = torch.tensor([[0.5, -0.2], [0.2, 0.1], [-0.1, 0.4]])
base = tl.trace(model, x)
base_loss = torch.nn.functional.mse_loss(base.output_ops[-1].out.detach(), target)
print("base loss", float(base_loss))
RESULT = float(base_loss) > 0

In [ ]:
# recipe-test: tool=torchlens expected=True
delta = torch.nn.Parameter(torch.zeros(4))
optimizer = torch.optim.SGD([delta], lr=0.25)
loss_history: list[float] = []


def add_delta(out: torch.Tensor, *, ctx: Any) -> torch.Tensor:
    """Add the learned vector to the activation."""
    return out + delta


for step in range(15):
    optimizer.zero_grad()
    edited = base.fork(f"train_step_{step}").do(tl.func("relu"), add_delta)
    loss = torch.nn.functional.mse_loss(edited.output_ops[-1].out, target)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

print([round(value, 4) for value in loss_history])
print("learned delta", delta.detach())
assert loss_history[-1] < loss_history[0]
RESULT = loss_history[-1] < loss_history[0]